# 20. Данные для прототипа (Kaggle)

Готовит компактные данные для приложения Streamlit (`src/app_data.py`): прогнозы тестового периода на час
и на сутки вперёд (гибрид, LightGBM, Chronos-2), интервалы по ошибкам на val, сценарии «что если» через LightGBM.

**Input:** датасет `vkr-transit-data`, вывод ноутбука 18 (в нём все прогнозы), вывод ноутбука `lgbm-final`
(файл модели `models/lgbm_final_trainval.txt`).
Accelerator — None, Internet — On. Время ~30 мин. Результат — `app_data.zip` в Output: распаковать в `app/data/`.

In [ ]:
import os, glob, shutil, subprocess, time
REPO = "/kaggle/working/vkr-transit"
if not os.path.exists(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Kinggl3/vkr-transit", REPO], check=True)
print(subprocess.run(["git", "-C", REPO, "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
lab = [os.path.dirname(p) for p in glob.glob("/kaggle/input/**/ridership_wide.parquet", recursive=True)]
assert lab, "Не найден ridership_wide.parquet: подключите датасет vkr-transit-data (Add Input)"
os.environ["VKR_LAB"] = lab[0]
os.environ["VKR_RESULTS"] = R = "/kaggle/working/results"
os.environ["PYTHONWARNINGS"] = "ignore"
os.makedirs(f"{R}/preds", exist_ok=True)
print("Данные:", lab[0], "| граф:", os.path.exists(os.path.join(os.path.dirname(lab[0]), "graph")),
      "| events.csv:", os.path.exists(os.path.join(lab[0], "events.csv")))
print("Подключено в /kaggle/input:")
for p in sorted(glob.glob("/kaggle/input/*/*")):
    print("  ", p)

In [ ]:
for p in sorted(glob.glob("/kaggle/input/**/results/preds/*.npz", recursive=True)):
    dst = f"{R}/preds/{os.path.basename(p)}"
    if not os.path.exists(dst):
        shutil.copy(p, dst)
need = [f"{m}__{s}.npz" for m in ("hyb_blend", "lgbm_final", "chronos2") for s in ("val", "test")]
missing = [n for n in need if not os.path.exists(f"{R}/preds/{n}")]
assert not missing, f"Нет прогнозов {missing}: подключите вывод ноутбука 18"
models = glob.glob("/kaggle/input/**/models/lgbm_final_trainval.txt", recursive=True)
assert models, "Нет models/lgbm_final_trainval.txt: подключите вывод ноутбука lgbm-final"
MODEL = models[0]
print(MODEL)

In [ ]:
!cd {REPO} && python src/app_data.py --model {MODEL} --out /kaggle/working/app_data

In [ ]:
import zipfile
with zipfile.ZipFile("/kaggle/working/app_data.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in glob.glob("/kaggle/working/app_data/*"):
        z.write(p, os.path.basename(p))
print(sorted((os.path.basename(p), round(os.path.getsize(p) / 2**20, 1)) for p in glob.glob("/kaggle/working/app_data/*")))